# Milestone 2 · Scoring: object recall, CLIPScore, KID, FID and diversity

**Owner:** Tanmayi · **Code:** shared `m2_scoring.py` (version `m2-score-v1`), usable on any member's image log

This notebook measures every image in my generation log with the group's shared scoring code. It reads only the files the generation notebook wrote (`images.csv` and the `images/` folder on Drive), so it runs in its own Colab session and never needs Stable Diffusion in memory.

| Measure | How | Reported as |
|---|---|---|
| Object recall | each labelled object (the `target` rows of `groundtruth_dev_500.csv`, 559 objects in 424 captions) is judged by two detectors: Faster R-CNN v2 (present at a class score ≥ 0.5) and OWL-ViT base-patch32 with the query "a photo of a {class}" (present at a score ≥ 0.1). An object is present when either judge finds it and missing when both miss it. | share of objects present, 95% interval |
| CLIPScore | 100 × max(cos(image, caption), 0), CLIP ViT-L/14 | mean, 95% interval |
| KID, FID | Inception-v3 pool features (the standard FID network) of the generated images against `ref_2000_v2`, 2,000 COCO val2017 photos that share no photo with `eval_2000` or either dev set | KID × 1000 with 95% interval; FID |
| Inception Score | exp(E KL(p(y\|x) ‖ p(y))) from the same Inception network's 1,008 class logits | IS, 95% interval |
| Real-photo floor | FID, KID and IS of the real COCO photos behind the same captions (none of them in `ref_2000_v2`), at the same number of images | next to every FID, KID and IS |
| Diversity | LPIPS (AlexNet) and 1 − CLIP image cosine between the seed_id 0 and seed_id 1 images of the same caption | mean, 95% interval |

Images blacked out by the safety checker are not scored. Intervals are 95% bootstrap intervals over captions (2,000 resamples, seed 0); with several seeds, a caption's seeds are resampled together. FID and IS depend on the number of images, so each is read next to the real-photo floor for the same number.

**Before running:** put `m2_scoring.py` (version `m2-score-v2`) in `MyDrive/m2/` next to `M2.zip`. Select a T4 GPU runtime. Scoring is incremental: images already scored are kept, so a rerun scores only new images (for example new seeds) and fills in anything added to the code since (the Inception Score logits, the real-photo floor).

## 1. Setup

In [1]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/m2"
SCORES = f"{DRIVE}/scores"            # every score file is written here
PHOTOS = "/content/coco_photos"      # real COCO photos, downloaded only when needed

!pip install -q torchmetrics torch-fidelity lpips
!unzip -q -o {DRIVE}/M2.zip -d /content
!cp {DRIVE}/m2_scoring.py /content/M2/protocol/

import sys, os
sys.path.insert(0, "/content/M2/protocol")
import pandas as pd
import m2_scoring as sc
print("scoring", sc.SCORING_VERSION)
assert sc.SCORING_VERSION == "m2-score-v2", ("copy the latest m2_scoring.py into MyDrive/m2/ first")

Mounted at /content/drive
scoring m2-score-v2


## 2. Which images to score

My own log, plus Santhosh's step-sweep log if he ran those images on his laptop (his `images.csv` and `images/` folder copied into `MyDrive/m2/steps_from_santhosh/`).

In [2]:
LOGS = [(f"{DRIVE}/images.csv", f"{DRIVE}/images")]
if os.path.exists(f"{DRIVE}/steps_from_santhosh/images.csv"):
    LOGS.append((f"{DRIVE}/steps_from_santhosh/images.csv", f"{DRIVE}/steps_from_santhosh/images"))

imgs = sc.read_logs(LOGS)
imgs.groupby(["split", "config_id", "seed_id"]).agg(images=("key", "size"), flagged=("flagged", "sum"))

images  flagged
split    config_id             seed_id                 
dev_500  ddim-leading-s10-g7.5 0           200        2
         ddim-leading-s20-g7.5 0           200        0
         ddim-leading-s30-g10  0           500        1
                               1           100        2
         ddim-leading-s30-g12  0           500        3
                               1           100        2
         ddim-leading-s30-g3   0           500        1
                               1           100        1
         ddim-leading-s30-g5   0           500        2
                               1           100        2
         ddim-leading-s30-g7.5 0           500        2
                               1           100        2
         ddim-leading-s50-g7.5 0           200        0
eval_500 ddim-leading-s10-g7.5 0           500        2
                               1           500        2
                               2           500        2
         ddim-leading-s30-g7.5 0           500        1
                               1           500        0
                               2           500        0

## 3. Real photos: the reference set, the floor and the judges' ceiling

- The 2,000 reference photos (`ref_2000_v2`) are needed once, for their Inception features.
- The 500 photos behind the `dev_500` captions, and the 500 behind the `eval_500` captions, give the **real-photo floor**: the FID, KID and IS that real photos get against the reference set with the same number of images. Neither set shares a photo with the reference set.
- The `dev_500` photos are also judged by the two judges: the recall they get on real photos is the ceiling a perfect generator could reach under these judges.

All photos get a centre square crop and are resized to 512 × 512, like the generated images. Everything is cached to Drive, so photos are downloaded only for what is still missing.

In [3]:
ref_ids = sc.load_split("ref_2000_v2").image_id
dev_ids = sc.load_split("dev_500").image_id
need_ref = not os.path.exists(f"{SCORES}/features/reference__ref_2000_v2.npz")
eval_ids = sc.load_split("eval_500").image_id
need_dev = not (os.path.exists(f"{SCORES}/objects/dev_500__real_photo.csv") and os.path.exists(f"{SCORES}/features/dev_500__real_photo.npz"))
need_eval = not os.path.exists(f"{SCORES}/features/eval_500__real_photo.npz")
to_get = (list(ref_ids) if need_ref else []) + (list(dev_ids) if need_dev else []) + (list(eval_ids) if need_eval else [])
if to_get:
    sc.download_coco(to_get, PHOTOS)     # about 2,500 photos, a few minutes

1000 photos in /content/coco_photos


In [4]:
scorers = sc.Scorers()     # each network loads when first needed: about 3 GB of GPU memory in all

sc.reference_features("ref_2000_v2", PHOTOS, SCORES, scorers)
sc.real_photo_features("dev_500", PHOTOS, SCORES, scorers)       # floor for dev_500
sc.real_photo_features("eval_500", PHOTOS, SCORES, scorers)      # floor for eval_500
sc.score_real_photos("dev_500", PHOTOS, SCORES, scorers)

real = sc.load_objects(SCORES, "dev_500").query("config_id == 'real_photo'")
r, lo, hi, n = sc.recall(real, "real_photo", "dev_500")
print(f"recall on the real dev photos: {r:.1%} [{lo:.1%}, {hi:.1%}] of {n} objects")
print("judge agreement on real photos:", {k: round(v, 3) for k, v in sc.judge_agreement(real).items()})

f_ref = sc.load_reference(SCORES)
dev_first200 = sc.load_split("dev_500").caption_idx.astype(str).head(200).tolist()
floor = pd.DataFrame([sc.real_floor(SCORES, "dev_500", f_ref), sc.real_floor(SCORES, "dev_500", f_ref, keys=dev_first200),
                      sc.real_floor(SCORES, "eval_500", f_ref)])
floor.to_csv(f"{SCORES}/real_photo_floor.csv", index=False)
floor.round(3)

reference features cached: /content/drive/MyDrive/m2/scores/features/reference__ref_2000_v2.npz


Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:00<00:00, 244MB/s]


real-photo features of dev_500: 500 photos -> /content/drive/MyDrive/m2/scores/features/dev_500__real_photo.npz
real-photo features of eval_500: 500 photos -> /content/drive/MyDrive/m2/scores/features/eval_500__real_photo.npz
real-photo verdicts cached: /content/drive/MyDrive/m2/scores/objects/dev_500__real_photo.csv
recall on the real dev photos: 98.9% [98.0%, 99.6%] of 559 objects
judge agreement on real photos: {'n': 559, 'agree': 0.93, 'kappa': 0.203, 'frcnn_only': 0.05, 'owl_only': 0.02}


,split,n,fid,kid,kid_lo,kid_hi,is,is_lo,is_hi
0,dev_500,500,78.022,0.053,-0.261,0.469,31.123,31.597,36.333
1,dev_500,200,132.761,-0.023,-0.708,0.742,24.741,26.031,30.861
2,eval_500,500,75.134,-0.160,-0.440,0.248,33.576,34.078,39.266


## 4. Score the dev_500 sweeps

About 0.3 s per image on a T4 (both detectors run on every image that has a labelled object): roughly 20 minutes for the guidance and step sweeps together. The Inception Score pass adds about 2 minutes.

In [5]:
sc.score_log(imgs, SCORES, scorers, splits=["dev_500"])
sc.score_diversity(imgs[imgs.split == "dev_500"], SCORES, scorers)
sc.score_inception_logits(imgs[imgs.split == "dev_500"], SCORES, scorers)

dev_500__ddim-leading-s30-g7.5: already scored
dev_500__ddim-leading-s30-g3: already scored
dev_500__ddim-leading-s30-g12: already scored
dev_500__ddim-leading-s30-g5: already scored
dev_500__ddim-leading-s30-g10: already scored
dev_500__ddim-leading-s10-g7.5: already scored
dev_500__ddim-leading-s20-g7.5: already scored
dev_500__ddim-leading-s50-g7.5: already scored
dev_500__ddim-leading-s30-g7.5: diversity already scored
dev_500__ddim-leading-s30-g3: diversity already scored
dev_500__ddim-leading-s30-g12: diversity already scored
dev_500__ddim-leading-s30-g5: diversity already scored
dev_500__ddim-leading-s30-g10: diversity already scored
dev_500__ddim-leading-s30-g7.5: logits for 596 images
dev_500__ddim-leading-s30-g3: logits for 598 images
dev_500__ddim-leading-s30-g12: logits for 595 images
dev_500__ddim-leading-s30-g5: logits for 596 images
dev_500__ddim-leading-s30-g10: logits for 597 images
dev_500__ddim-leading-s10-g7.5: logits for 198 images
dev_500__ddim-leading-s20-g7.5: l

## 5. Summary table for dev_500

In [6]:
summary = sc.summary_table(SCORES, split="dev_500")
summary.to_csv(f"{SCORES}/summary_dev_500_all_captions.csv", index=False)
summary[["config_id", "images", "flagged", "objects", "recall", "recall_lo", "recall_hi", "clipscore", "kid", "fid", "is",
         "floor_n", "floor_kid", "floor_fid", "floor_is", "sec_per_image"]].round(3)

,config_id,images,flagged,objects,recall,recall_lo,recall_hi,clipscore,kid,fid,is,floor_n,floor_kid,floor_fid,floor_is,sec_per_image
0,ddim-leading-s10-g7.5,200,2,218,0.940,0.908,0.970,27.006,3.419,141.592,23.532,198,-0.048,133.331,24.893,1.889
1,ddim-leading-s20-g7.5,200,0,221,0.937,0.902,0.968,26.809,4.242,136.225,28.508,200,-0.023,132.761,24.741,3.515
2,ddim-leading-s30-g10,500,1,559,0.928,0.907,0.948,26.867,5.641,86.202,32.086,499,0.062,78.178,31.155,5.480
3,ddim-leading-s30-g12,500,3,556,0.937,0.916,0.956,26.853,6.025,86.932,30.641,497,0.050,78.352,31.085,5.481
4,ddim-leading-s30-g3,500,1,558,0.918,0.892,0.941,25.998,1.685,81.795,32.662,499,0.049,78.097,31.096,5.491
5,ddim-leading-s30-g5,500,2,556,0.939,0.919,0.959,26.734,3.538,83.290,33.484,498,0.037,78.191,31.139,5.487
6,ddim-leading-s30-g7.5,500,2,557,0.937,0.917,0.957,26.760,4.350,85.027,31.853,498,0.039,78.211,31.083,5.478
7,ddim-leading-s50-g7.5,200,0,221,0.937,0.901,0.967,26.801,4.434,135.578,26.720,200,-0.023,132.761,24.741,8.398


## 6. Final numbers on eval_500 (run after the generation notebook's section 7)

These are scored after the setting has been chosen on `dev_500`: all three seeds of the anchor and of the chosen setting. The table shows seed_id 0; the analysis notebook pools the three seeds.

In [8]:
imgs = sc.read_logs(LOGS)
if (imgs.split == "eval_500").any():
    sc.score_log(imgs, SCORES, scorers, splits=["eval_500"])
    sc.score_inception_logits(imgs[imgs.split == "eval_500"], SCORES, scorers)
    final = sc.summary_table(SCORES, split="eval_500")
    final.to_csv(f"{SCORES}/summary_eval_500.csv", index=False)
    display(imgs[imgs.split == "eval_500"].groupby(["config_id", "seed_id"]).agg(images=("key", "size"), flagged=("flagged", "sum")))
    display(final[["config_id", "images", "flagged", "objects", "recall", "recall_lo", "recall_hi", "clipscore", "kid", "fid", "is",
                   "floor_kid", "floor_fid", "floor_is"]].round(3))
else:
    print("no eval_500 images yet: run the generation notebook's section 7 first")

eval_500__ddim-leading-s30-g7.5: already scored
eval_500__ddim-leading-s10-g7.5: already scored
eval_500__ddim-leading-s30-g7.5: logits already computed
eval_500__ddim-leading-s10-g7.5: logits for 1494 images


images  flagged
config_id             seed_id                 
ddim-leading-s10-g7.5 0           500        2
                      1           500        2
                      2           500        2
ddim-leading-s30-g7.5 0           500        1
                      1           500        0
                      2           500        0

,config_id,images,flagged,objects,recall,recall_lo,recall_hi,clipscore,kid,fid,is,floor_kid,floor_fid,floor_is
0,ddim-leading-s10-g7.5,500,2,566,0.940,0.921,0.958,26.650,3.210,84.998,32.301,-0.160,75.381,33.573
1,ddim-leading-s30-g7.5,500,1,567,0.949,0.931,0.966,26.597,4.986,84.788,35.961,-0.155,75.244,33.661


## For teammates: scoring your own images

```python
!pip install -q torchmetrics torch-fidelity lpips
import sys; sys.path.insert(0, "M2/protocol")        # with m2_scoring.py copied into M2/protocol
import m2_scoring as sc

imgs = sc.read_logs([("<your log folder>/images.csv", "<your log folder>/images")])
scorers = sc.Scorers()
sc.score_log(imgs, "<your log folder>/scores", scorers)          # dev_500, eval_500, eval_2000 or stress_prompts
sc.score_diversity(imgs, "<your log folder>/scores", scorers)      # only where seed_id 1 pairs exist
sc.score_inception_logits(imgs, "<your log folder>/scores", scorers)   # Inception Score
# copy features/reference__ref_2000_v2.npz (and dev_500__real_photo.npz, eval_500__real_photo.npz for the floor)
# from MyDrive/m2/scores/features/ into <your log folder>/scores/features/
table = sc.summary_table("<your log folder>/scores")
```

Stress prompts are judged on the objects named in `stress_prompts.csv`; Faster R-CNN only knows the 80 COCO classes, so an object outside them is judged by OWL-ViT alone. `objects/<split>__<config>.csv` keeps both judges' raw scores, so the thresholds can be checked later without rescoring.